# Notebook 13 — Null Handling

**Datasets:** `samples.bakehouse.sales_customers`, `samples.bakehouse.sales_franchises`, `samples.tpch.customer`  
**Difficulty:** Easy  
**Topics:** `isNull`, `isNotNull`, `fillna`, `dropna`, `coalesce`, `when/otherwise` for nulls, counting nulls per column

Null handling is one of the most critical skills in data engineering. These problems teach the standard patterns.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = SparkSession.builder.getOrCreate()

customers = spark.read.table("samples.bakehouse.sales_customers")
franchises = spark.read.table("samples.bakehouse.sales_franchises")
tpch_customer = spark.read.table("samples.tpch.customer")

print("sales_customers schema:")
customers.printSchema()
print("sales_franchises schema:")
franchises.printSchema()
print("tpch.customer schema:")
tpch_customer.printSchema()

## Learn — Null Handling

| Function | What it does |
|----------|-------------|
| `F.col("c").isNull()` | True when value is null |
| `F.col("c").isNotNull()` | True when value is not null |
| `df.dropna(subset=["col"])` | Drop rows where the specified column is null |
| `df.fillna({"col": default})` | Replace nulls with a default value |
| `F.coalesce(col1, col2, ...)` | Return the first non-null value from the list |
| `F.when(F.col("c").isNull(), "missing").otherwise(F.col("c"))` | Replace null with a label |
| `F.count("col")` | Counts non-null values (use `F.count("*")` to count all rows) |

**Docs:** [DataFrame.fillna()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.fillna.html) · [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

> `F.count("col_name")` skips nulls. `F.count("*")` counts every row including nulls.
> This difference is important when auditing data quality.

In [ ]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.bakehouse.sales_customers")

# Count nulls in each column — useful for data quality checks
from pyspark.sql import functions as F
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show()

# coalesce: return city, or country if city is null, or "Unknown" if both null
df.select(
    F.coalesce(F.col("city"), F.col("country"), F.lit("Unknown")).alias("location")
).show(5)

## Problem 1 — Count Nulls Per Column

On `sales_customers`, compute the number of null values for **every** column. Loop over `df.columns` and use `F.sum(F.col(c).isNull().cast("int"))` to count nulls per column. Also compute `null_rate_pct` as the percentage of rows that are null.


**Expected output columns:** `column_name`, `null_count`, `null_rate_pct`

In [ ]:
result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ─────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'column_name' in cols, "Missing column: column_name"
assert 'null_count' in cols, "Missing column: null_count"
assert 'null_rate_pct' in cols, "Missing column: null_rate_pct"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2 — Filter Nulls

On `sales_customers`, find:
1. Customers where `state` **is null** using `.filter(F.col("state").isNull())`
2. Customers where `state` **is not null** using `.filter(F.col("state").isNotNull())`

Return a single summary DataFrame showing both counts.

**Expected output columns:** `filter_type`, `customer_count`

In [ ]:
result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ─────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'filter_type' in cols, "Missing column: filter_type"
assert 'customer_count' in cols, "Missing column: customer_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3 — Fill Nulls

On `sales_customers`, use `.fillna()` to replace null values with sensible defaults:
- `state` → `"Unknown"`
- `gender` → `"Not Specified"`

Return the fully cleaned DataFrame with nulls replaced.


**Expected output columns:** all original columns of `sales_customers` (with nulls filled in `state` and `gender`)

In [ ]:
result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ─────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'state' in cols, "Missing column: state"
assert 'gender' in cols, "Missing column: gender"
cnt = result_3.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4 — Coalesce for Null Fallback

On `sales_franchises`, create a `location_label` column using `F.coalesce()` that falls back through:
1. `district` (use if not null)
2. `city` (use if district is null)
3. `"Unknown"` (literal fallback via `F.lit()`)


**Expected output columns:** `franchiseID`, `name`, `district`, `city`, `location_label`

In [ ]:
result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ─────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'district' in cols, "Missing column: district"
assert 'city' in cols, "Missing column: city"
assert 'location_label' in cols, "Missing column: location_label"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5 — Drop Rows with Nulls

On `tpch.customer`, use `.dropna(subset=["c_name", "c_mktsegment"])` to remove rows where either key column is null. Show the row count **before** and **after** in a summary DataFrame.


**Expected output columns:** `step`, `row_count`

In [ ]:
result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ─────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'step' in cols, "Missing column: step"
assert 'row_count' in cols, "Missing column: row_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 5 passed ✓  ({cnt} rows)")